# Práctica RAG - Fase 3: Benchmark de Modelos LLM

En este notebook ejecutamos la **Fase 3** de la práctica: comparar 4 modelos de lenguaje locales (LLMs) con **Ollama** sobre los dos documentos del proyecto, utilizando los fragmentos recuperados por los embeddings ganadores de la Fase 2.

### Los dos documentos evaluados:
1. **Normativa jurídica:** *Estatuto de los Trabajadores* (BOE consolidado, RDL 2/2015).
2. **Documentación técnica:** *Manual del Administrador de Debian* (Debian 11 Bullseye).

### Los cuatro modelos LLM candidatos:
- **Modelos pequeños (~2.6B):**
  - **`Gemma-2-2B`** (`gemma2:2b`, Google DeepMind, 2.6B, Q4_0): modelo ligero y rápido de Google.
  - **`Granite-3-2B`** (`granite3-dense:2b`, IBM, 2.6B, Q4_K_M): modelo compacto de IBM pensado para tareas de empresa y RAG.
- **Modelos medianos (~7B):**
  - **`Mistral-7B`** (`mistral:7b`, Mistral AI, 7.2B, Q4_K_M): referencia abierta muy equilibrada en razonamiento y seguimiento de instrucciones.
  - **`Qwen-2.5-7B`** (`qwen2.5:7b`, Alibaba Cloud, 7.6B, Q4_K_M): modelo multilingüe reciente con muy buen nivel técnico.

In [1]:
# Librerías para comunicarnos con Ollama, procesar datos y mostrar tablas.
import json
import re
import time
import os
import urllib.request
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from IPython.display import display, Markdown

# Comprobamos si hay GPU y consultamos qué modelos tiene Ollama descargados.
# Ollama debe estar arrancado en este ordenador para responder a esta dirección.
# Se usa GPU para los embeddings si está disponible; Ollama administra sus modelos aparte.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo de cómputo en uso: {device}')
if device == 'cuda':
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}')
    print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

# Esta dirección local es la API de Ollama; /api/tags devuelve los modelos instalados.
req = urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=5)
modelos_ollama = json.loads(req.read().decode())['models']
print(f'Modelos disponibles en Ollama ({len(modelos_ollama)}):')
for m in modelos_ollama:
    print(f"  - {m['name']} ({m['details']['parameter_size']}, {m['details']['quantization_level']})")

/home/g4/reto_1_2-2627-Rag-Benchmark/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo de cómputo en uso: cuda
GPU detectada: NVIDIA A800 40GB Active MIG 3g.20gb
VRAM disponible: 19.62 GB
Modelos disponibles en Ollama (4):
  - qwen2.5:7b (7.6B, Q4_K_M)
  - mistral:7b (7.2B, Q4_K_M)
  - granite3-dense:2b (2.6B, Q4_K_M)
  - gemma2:2b (2.6B, Q4_0)


## 1. Definición de Candidatos, Documentos y System Prompt

Para que la comparativa sea justa, los 4 modelos reciben los mismos fragmentos recuperados (top-3) y exactamente el mismo **system prompt** con una plantilla clara:
1. Responder solo con lo que dice el texto proporcionado.
2. Si el dato no aparece, responder exactamente: *«No aparece en el documento»*.
3. Si la respuesta sí aparece, indicar primero la fuente (`Fuente: ...`) y después la respuesta en español (`Respuesta: ...`).

In [2]:
# Modelos de lenguaje que se van a comparar y algunos datos para identificarlos.
LLM_MODELOS = [
    {
        # Este nombre debe coincidir con el que reconoce Ollama.
        'nombre': 'gemma2:2b',
        'alias': 'Gemma-2-2B',
        # Los campos restantes sirven para describir el modelo en las tablas.
        'categoria': 'Pequeño',
        'params': '2.6B',
        'cuantizacion': 'Q4_0',
        'stack': 'Ollama'
    },
    {
        'nombre': 'granite3-dense:2b',
        'alias': 'Granite-3-2B',
        'categoria': 'Pequeño',
        'params': '2.6B',
        'cuantizacion': 'Q4_K_M',
        'stack': 'Ollama'
    },
    {
        'nombre': 'mistral:7b',
        'alias': 'Mistral-7B',
        'categoria': 'Mediano',
        'params': '7.2B',
        'cuantizacion': 'Q4_K_M',
        'stack': 'Ollama'
    },
    {
        'nombre': 'qwen2.5:7b',
        'alias': 'Qwen-2.5-7B',
        'categoria': 'Mediano',
        'params': '7.6B',
        'cuantizacion': 'Q4_K_M',
        'stack': 'Ollama'
    }
]

# Ajustes para cada documento, incluido el modelo que busca los fragmentos relevantes.
DOCUMENTOS = [
    {
        'id': 'boe',
        'nombre': '1. Normativa (BOE Estatuto de los Trabajadores)',
        'archivo_chunks': 'data/processed/boe_chunks.json',
        'archivo_preguntas': 'preguntas/boe_questions.json',
        'embedding_model': 'google/embeddinggemma-2',
        'prefix_query': '',
        'prefix_doc': '',
        'trust_remote_code': True
    },
    {
        'id': 'debian',
        'nombre': '2. Técnica (Debian Handbook)',
        'archivo_chunks': 'data/processed/debian_chunks.json',
        'archivo_preguntas': 'preguntas/debian_questions.json',
        'embedding_model': 'intfloat/multilingual-e5-small',
        'prefix_query': 'query: ',
        'prefix_doc': 'passage: ',
        'trust_remote_code': False
    }
]

# Instrucciones que recibirá el modelo en todas las preguntas.
SYSTEM_PROMPT = """Eres un asistente riguroso especializado en responder preguntas sobre documentos en español.
Sigue estas reglas de forma estricta:
1. Basa tu respuesta ÚNICAMENTE en los fragmentos de contexto proporcionados. No uses conocimientos externos ni inventes datos.
2. Si la respuesta no aparece en los fragmentos, responde exactamente: No aparece en el documento.
3. Si la respuesta sí aparece en los fragmentos, responde en español empezando SIEMPRE por indicar la fuente exacta (el artículo, capítulo o sección que aparece entre corchetes), usando este formato:
Fuente: [artículo, capítulo o sección del fragmento]
Respuesta: [tu respuesta clara y completa en español]"""

# Plantilla que combina los fragmentos recuperados y la pregunta que se hará al LLM.
USER_TEMPLATE = """Fragmentos de contexto:
{contexto}

Pregunta:
{pregunta}"""

## 2. Recuperación de Contexto (Top-3 Fragmentos)

Recuperamos los 3 fragmentos más relevantes para cada una de las 60 preguntas (30 del BOE y 30 de Debian) usando el embedding ganador de cada documento en la Fase 2. Después liberamos la GPU para que Ollama tenga toda la memoria libre.

In [3]:
# Busca de antemano los tres fragmentos más parecidos a cada pregunta.
def precomputar_contextos():
    contextos_por_doc = {}
    for doc in DOCUMENTOS:
        # Abrimos los fragmentos y las preguntas correspondientes a este documento.
        with open(doc['archivo_chunks'], 'r', encoding='utf-8') as f:
            chunks = json.load(f)
        with open(doc['archivo_preguntas'], 'r', encoding='utf-8') as f:
            preguntas = json.load(f)

        print(f"Indexando {doc['nombre']} con {doc['embedding_model']}...")
        # Este modelo de embeddings se usa para buscar contexto; no es el LLM que responde.
        model_emb = SentenceTransformer(
            doc['embedding_model'],
            device=device,
            trust_remote_code=doc['trust_remote_code']
        )

        # Convertimos los textos del documento en vectores numéricos para compararlos.
        doc_texts = [doc['prefix_doc'] + c['text'] for c in chunks]
        # encode transforma los textos en vectores; normalizarlos facilita comparar similitudes.
        doc_embs = model_emb.encode(
            doc_texts,
            batch_size=32,
            show_progress_bar=False,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        # Para cada pregunta guardaremos su texto y los tres fragmentos recuperados.
        preguntas_con_contexto = []
        for q in preguntas:
            # Añadimos el prefijo que necesita el modelo y convertimos la pregunta en vector.
            q_text = doc['prefix_query'] + q['pregunta']
            q_emb = model_emb.encode(q_text, convert_to_tensor=True, normalize_embeddings=True)
            # Comparamos la pregunta con todos los fragmentos y elegimos los 3 mejores.
            scores = util.cos_sim(q_emb, doc_embs)[0]
            topk_idx = torch.topk(scores, k=min(3, len(chunks))).indices.tolist()

            # Los índices son posiciones de la lista original de fragmentos.
            top_chunks = [chunks[i] for i in topk_idx]
            # Unimos los fragmentos elegidos para pasárselos al modelo de lenguaje.
            contexto_str = '\n\n---\n\n'.join([f"[{c['title']}]\n{c['text']}" for c in top_chunks])
            preguntas_con_contexto.append({
                'pregunta_info': q,
                'top_chunks': [c['id'] for c in top_chunks],
                'contexto': contexto_str
            })

        # Quitamos el modelo de embeddings y sus vectores de memoria al terminar el documento.
        del model_emb
        del doc_embs
        if device == 'cuda':
            torch.cuda.empty_cache()

        contextos_por_doc[doc['id']] = preguntas_con_contexto
        print(f"  -> {len(preguntas_con_contexto)} preguntas listas con contexto top-3.")

    return contextos_por_doc

contextos_por_doc = precomputar_contextos()

Indexando 1. Normativa (BOE Estatuto de los Trabajadores) con google/embeddinggemma-2...


Could not reach the Hub (Server disconnected without sending a response.). Using cached commit hash for 'google/embeddinggemma-2'.


Loading weights:   0%|                                | 0/1376 [00:00<?, ?it/s]

Loading weights:  61%|████████████▏       | 836/1376 [00:00<00:00, 8342.35it/s]

Loading weights: 100%|███████████████████| 1376/1376 [00:00<00:00, 7547.35it/s]

  -> 30 preguntas listas con contexto top-3.
Indexando 2. Técnica (Debian Handbook) con intfloat/multilingual-e5-small...


Loading weights:   0%|                                 | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 6507.97it/s]

  -> 30 preguntas listas con contexto top-3.


## 3. Funciones de Inferencia, Velocidad y Rúbrica de Calidad

Definimos las funciones que consultan Ollama, liberan VRAM entre modelos, miden las métricas de coste en caliente (TTFT, tokens/s, latencia y VRAM) y evalúan cada respuesta según la rúbrica:
- **Corrección (0–2):** 2 = correcta y completa, 1 = parcial, 0 = incorrecta (en preguntas sin respuesta, decir «No aparece en el documento» puntúa 2).
- **Fidelidad:** no inventa datos ajenos al contexto.
- **Abstención correcta / Falsas abstenciones:** comprueba si sabe callar cuando el dato no está y responder cuando sí está.
- **Formato e idioma:** cita la fuente (artículo, capítulo o sección) y responde en español.

In [4]:
# Palabras demasiado comunes que no ayudan a comprobar si una respuesta es correcta.
STOPWORDS = {
    'como', 'para', 'pero', 'este', 'esta', 'estos', 'estas', 'sobre', 'entre',
    'puede', 'pueden', 'utiliza', 'mediante', 'cuando', 'donde', 'todos', 'todas',
    'tiene', 'tienen', 'debe', 'deben', 'hace', 'solo', 'son', 'los', 'las', 'del',
    'por', 'con', 'una', 'uno', 'que', 'sus', 'sin', 'mas', 'más', 'ser', 'sido',
    'articulo', 'artículo', 'seccion', 'sección', 'capitulo', 'capítulo', 'documento'
}

# Envía una pregunta a Ollama y espera a que el modelo termine de responder.
def llamar_ollama(model_name, user_prompt, system_prompt=SYSTEM_PROMPT, temperature=0.0, num_predict=256, timeout=300):
    # La API recibe el nombre del modelo, las instrucciones y la pregunta en formato JSON.
    # stream=False hace que devuelva la respuesta completa de una vez.
    req = urllib.request.Request(
        'http://127.0.0.1:11434/api/generate'    req = urllib.request.Request(
        'http://127.0.0.1:11434/api/generate',
        headers={'Content-Type': 'application/json'},
        data=json.dumps({
            'model': model_name,
            'system': system_prompt,
            'prompt': user_prompt,
            'stream': False,
            'options': {
                'temperature': temperature,
                'num_predict': num_predict
            }
        }).encode('utf-8')
    )
    return json.loads(urllib.request.urlopen(req, timeout=timeout).read().decode())

# Pide a Ollama que libere los modelos que siguen ocupando memoria.
def descargar_modelos_activos():
    try:
        req = urllib.request.Request('http://127.0.0.1:11434/api/ps')
        data = json.loads(urllib.request.urlopen(req, timeout=10).read().decode())
        for m in data.get('models', []):
            req_u = urllib.request.Request(
                'http://127.0.0.1:11434/api/generate',
                headers={'Content-Type': 'application/json'},
                data=json.dumps({'model': m.get('name'), 'keep_alive': 0}).encode('utf-8')
            )
            urllib.request.urlopen(req_u, timeout=10)
        time.sleep(1)
    # Si Ollama no responde, seguimos con el benchmark sin detener todo el notebook.
    except Exception:
        pass    except Exception:
        pass

# Consulta cuánta memoria de la GPU está usando el modelo.
def obtener_vram_ollama(model_name):
    try:
        req = urllib.request.Request('http://127.0.0.1:11434/api/ps')
        data = json.loads(urllib.request.urlopen(req, timeout=10).read().decode())
        for m in data.get('models', []):
            if model_name in m.get('name', ''):
                return round(m.get('size_vram', 0) / (1024**3), 2)
    except Exception:
        pass
    return 0.0

# Hace una evaluación automática sencilla de la respuesta del modelo.
# Comprueba si responde cuando debe, si se abstiene cuando falta información y si menciona palabras clave.
def evaluar_respuesta_calidad(resp_text, pregunta_info):
    resp_lower = resp_text.lower().strip()
    es_sin_respuesta = pregunta_info['tipo'] == 'sin_respuesta'
    dice_no_aparece = (
        'no aparece' in resp_lower
        or 'no se menciona' in resp_lower
        or 'no figura' in resp_lower
        or 'no está' in resp_lower
        or 'no se encuentra' in resp_lower
    )

    abstencion_correcta = bool(es_sin_respuesta and dice_no_aparece)
    falsa_abstencion = bool((not es_sin_respuesta) and dice_no_aparece)

    cita_fuente = (
        'art' in resp_lower
        or 'sección' in resp_lower
        or 'seccion' in resp_lower
        or 'capítulo' in resp_lower
        or 'capitulo' in resp_lower
        or 'fuente:' in resp_lower
        or dice_no_aparece
    )

    # 'sin_respuesta' identifica preguntas cuya respuesta no está en el documento.
    if es_sin_respuesta:    if es_sin_respuesta:
        correccion = 2 if dice_no_aparece else 0
        fiel = dice_no_aparece
    else:
        if falsa_abstencion:
            correccion = 0
            fiel = True
        else:
            resp_esperada = pregunta_info['respuesta_esperada'].lower()
            # Separamos la respuesta esperada en palabras para buscar cuáles aparecen
            # en la respuesta generada.
            tokens = re.findall(r'[a-zA-Z0-9áéíóúñÁÉÍÓÚÑ/_-]+', resp_esperada)
            palabras_clave = [
                w for w in tokens
                if (len(w) >= 4 or any(ch.isdigit() for ch in w) or w in {'apt', 'ssh', 'ufw', 'ftp', 'dns'})
                and w not in STOPWORDS
            ]
            # Contamos qué proporción de palabras clave aparece literalmente en la respuesta.
            coincidencias = sum(1 for w in palabras_clave if w in resp_lower)
            ratio = coincidencias / max(1, len(palabras_clave))

            # Se asigna 0, 1 o 2 según la proporción de palabras clave encontradas.
            if ratio >= 0.40:
                correccion = 2
            elif ratio >= 0.20:
                correccion = 1
            else:
                correccion = 0
            fiel = (correccion > 0)

    return {
        'correccion': correccion,
        'fiel': fiel,
        'abstencion_correcta': abstencion_correcta,
        'falsa_abstencion': falsa_abstencion,
        'formato_idioma': cita_fuente
    }

# Mide cuánto tarda el modelo en responder y cuántos tokens genera por segundo.
def medir_coste_modelo(modelo_nombre):
    # Esta primera pregunta sirve para preparar el modelo antes de medirlo.
    prompt_warmup = USER_TEMPLATE.format(
        contexto='[Artículo 38. Vacaciones anuales]\nLas vacaciones anuales retribuidas son de 30 días naturales.',
        pregunta='¿Cuántos días de vacaciones corresponden?'
    )
    llamar_ollama(modelo_nombre, prompt_warmup)

    prompts_prueba = [
        USER_TEMPLATE.format(contexto='[Artículo 14. Periodo de prueba]\nEl periodo de prueba tiene un máximo de 6 meses para técnicos titulados.', pregunta='¿Cuánto dura el periodo de prueba?'),
        USER_TEMPLATE.format(contexto='[Sección 6.2. APT]\nLa orden apt update se usa para refrescar los repositorios en Debian.', pregunta='¿Qué orden actualiza los paquetes?'),
        USER_TEMPLATE.format(contexto='[Artículo 35. Horas extraordinarias]\nEl número de horas extraordinarias no podrá ser superior a 80 al año.', pregunta='¿Cuántas horas extras se pueden hacer?'),
        USER_TEMPLATE.format(contexto='[Sección 9.2. SSH]\nLa configuración del servidor SSH se encuentra en /etc/ssh/sshd_config.', pregunta='¿Dónde está la configuración de SSH?'),
        USER_TEMPLATE.format(contexto='[Artículo 6. Trabajo de los menores]\nSe prohíbe la admisión al trabajo a los menores de dieciséis años.', pregunta='¿Cuál es la edad mínima laboral?')
    ]

    # Repetimos varias veces cada prueba y luego usamos la mediana para resumir tiempos.
    ttfts, tokens_per_secs, latencias = [], [], []
    for p in prompts_prueba:
        p_ttfts, p_tps, p_lats = [], [], []
        for _ in range(3):
            t0 = time.time()
            res = llamar_ollama(modelo_nombre, p)
            p_lats.append(time.time() - t0)
            # Ollama informa duraciones en nanosegundos; dividimos por mil millones
            # para expresarlas en segundos.
            p_ttfts.append(res.get('prompt_eval_duration', 0) / 1e9)
            eval_dur = res.get('eval_duration', 0) / 1e9
            eval_cnt = res.get('eval_count', 0)
            if eval_dur > 0:
                p_tps.append(eval_cnt / eval_dur)

        latencias.append(float(np.median(p_lats)))
        ttfts.append(float(np.median(p_ttfts)))
        if p_tps:
            tokens_per_secs.append(float(np.median(p_tps)))

    # La mediana reduce el efecto de una ejecución especialmente lenta o rápida.
    return {
        'ttft_s'    return {
        'ttft_s': round(float(np.median(ttfts)), 3),
        'tokens_per_s': round(float(np.median(tokens_per_secs)), 1),
        'latencia_s': round(float(np.median(latencias)), 2),
        'vram_gb': obtener_vram_ollama(modelo_nombre)
    }

## 4. Ejecución del Benchmark Completo y Guardado de Resultados

Evaluamos los 4 modelos en coste y calidad sobre las 60 preguntas de ambos documentos y guardamos los datos brutos en `resultados/`.

In [5]:
# Creamos la carpeta de resultados si todavía no existe.
os.makedirs('resultados', exist_ok=True)
tabla_b_calidad = []
tabla_c_coste = []
respuestas_detalladas = []

print('=' * 70)
print('INICIANDO BENCHMARK DE LLMS PARA RAG (FASE 3)')
print('=' * 70)

# Para cada modelo medimos primero velocidad y después calidad de las respuestas.
for m in LLM_MODELOS:
    print(f"\n>>> Evaluando modelo: {m['alias']} ({m['nombre']})")
    # Liberamos modelos anteriores antes de cargar el siguiente para controlar la memoria.
    descargar_modelos_activos()

    # 1. Medición de coste y velocidad (Tabla C)
    coste_info = medir_coste_modelo(m['nombre'])
    tabla_c_coste.append({
        'modelo': m['alias'],
        'stack': m['stack'],
        'ttft_s': coste_info['ttft_s'],
        'tokens_per_s': coste_info['tokens_per_s'],
        'latencia_s': coste_info['latencia_s'],
        'vram_gb': coste_info['vram_gb']
    })
    print(f"  Coste -> {coste_info['tokens_per_s']} tok/s | Latencia: {coste_info['latencia_s']}s | TTFT: {coste_info['ttft_s']}s | VRAM: {coste_info['vram_gb']} GB")

    # 2. Medición de calidad sobre todas las preguntas (Tabla B)
    eval_scores = []
    sin_resp_correctas = 0
    total_sin_resp = 0
    falsas_abs = 0
    fieles_count = 0
    formato_count = 0

    # Probamos el modelo con las preguntas y contextos preparados para cada documento.
    for doc_id, items in contextos_por_doc.items():
        for item in items:
            q_info = item['pregunta_info']
            user_prompt = USER_TEMPLATE.format(contexto=item['contexto'], pregunta=q_info['pregunta'])
            # El modelo recibe el contexto recuperado y la pregunta original.
            res_ollama = llamar_ollama(m['nombre'], user_prompt)
            respuesta_salida = res_ollama.get('response', '').strip()

            # Aplicamos las reglas automáticas definidas arriba a la respuesta recibida.
            eval_res = evaluar_respuesta_calidad(respuesta_salida, q_info)
            eval_scores.append(eval_res['correccion'])
            if eval_res['fiel']:
                fieles_count += 1
            if q_info['tipo'] == 'sin_respuesta':
                total_sin_resp += 1
            if eval_res['abstencion_correcta']:
                sin_resp_correctas += 1
            if eval_res['falsa_abstencion']:
                falsas_abs += 1
            if eval_res['formato_idioma']:
                formato_count += 1

            # Conservamos también cada respuesta individual para poder revisarla luego.
            respuestas_detalladas.append({
                'modelo': m['alias'],
                'documento': doc_id,
                'pregunta_id': q_info['id'],
                'pregunta': q_info['pregunta'],
                'tipo': q_info['tipo'],
                'respuesta_modelo': respuesta_salida,
                'respuesta_esperada': q_info['respuesta_esperada'],
                'evaluacion': eval_res
            })

    # Calculamos porcentajes y medias a partir de las evaluaciones de las preguntas.
    total_preguntas = len(eval_scores)
    correccion_media = round(float(np.mean(eval_scores)), 2)
    fieles_pct = round((fieles_count / total_preguntas) * 100, 1)
    formato_pct = round((formato_count / total_preguntas) * 100, 1)

    tabla_b_calidad.append({
        'modelo': m['alias'],
        'tamano': m['params'],
        'cuantizacion': m['cuantizacion'],
        'correccion_media': correccion_media,
        'respuestas_fieles_pct': fieles_pct,
        'abstencion_correcta': f'{sin_resp_correctas}/{total_sin_resp}',
        'falsas_abstenciones': falsas_abs,
        'formato_idioma_pct': formato_pct
    })
    print(f"  Calidad -> Corrección: {correccion_media}/2.0 | Fieles: {fieles_pct}% | Abstención: {sin_resp_correctas}/{total_sin_resp} | Falsas abs.: {falsas_abs} | Formato: {formato_pct}%")

# Al terminar, volvemos a liberar los modelos que Ollama pudiera mantener cargados.
descargar_modelos_activos()

# Guardamosdescargar_modelos_activos()

# Guardamos por separado el resumen y las respuestas completas de cada modelo.
datos_benchmark = {
    'tabla_b_calidad': tabla_b_calidad,
    'tabla_c_coste': tabla_c_coste
}
with open('resultados/llms_benchmark.json', 'w', encoding='utf-8') as f:
    json.dump(datos_benchmark, f, ensure_ascii=False, indent=2)

with open('resultados/llms_respuestas_detalladas.json', 'w', encoding='utf-8') as f:
    json.dump(respuestas_detalladas, f, ensure_ascii=False, indent=2)

print('\nResultados guardados correctamente en resultados/llms_benchmark.json y resultados/llms_respuestas_detalladas.json')

INICIANDO BENCHMARK DE LLMS PARA RAG (FASE 3)

>>> Evaluando modelo: Gemma-2-2B (gemma2:2b)


  Coste -> 86.7 tok/s | Latencia: 0.44s | TTFT: 0.044s | VRAM: 1.52 GB


  Calidad -> Corrección: 1.7/2.0 | Fieles: 98.3% | Abstención: 11/12 | Falsas abs.: 6 | Formato: 98.3%

>>> Evaluando modelo: Granite-3-2B (granite3-dense:2b)


  Coste -> 98.9 tok/s | Latencia: 0.41s | TTFT: 0.015s | VRAM: 1.49 GB


  Calidad -> Corrección: 1.68/2.0 | Fieles: 98.3% | Abstención: 11/12 | Falsas abs.: 4 | Formato: 75.0%

>>> Evaluando modelo: Mistral-7B (mistral:7b)


  Coste -> 72.9 tok/s | Latencia: 0.68s | TTFT: 0.016s | VRAM: 4.07 GB


  Calidad -> Corrección: 1.75/2.0 | Fieles: 100.0% | Abstención: 12/12 | Falsas abs.: 6 | Formato: 100.0%

>>> Evaluando modelo: Qwen-2.5-7B (qwen2.5:7b)


  Coste -> 74.0 tok/s | Latencia: 0.43s | TTFT: 0.014s | VRAM: 4.36 GB


  Calidad -> Corrección: 1.73/2.0 | Fieles: 100.0% | Abstención: 12/12 | Falsas abs.: 8 | Formato: 100.0%



Resultados guardados correctamente en resultados/llms_benchmark.json y resultados/llms_respuestas_detalladas.json


## 5. Visualización Formateada: Tabla B (Calidad) y Tabla C (Coste)

In [6]:
# Estas tablas muestran un resumen de calidad y de velocidad de los modelos.
display(Markdown('### Tabla B. LLMs: Calidad de Respuestas RAG'))
# Convertimos los resultados de calidad en una tabla y elegimos las columnas visibles.
df_calidad = pd.DataFrame(tabla_b_calidad)
df_b_view = df_calidad[[
    'modelo', 'tamano', 'cuantizacion',
    'correccion_media', 'respuestas_fieles_pct',
    'abstencion_correcta', 'falsas_abstenciones', 'formato_idioma_pct'
]].copy()
df_b_view.columns = [
    'Modelo', 'Tamaño', 'Cuantización',
    'Corrección (0-2)', '% Fieles',
    'Abstención Correcta', 'Falsas Abstenciones', '% Formato e Idioma'
]
# Estos formatos controlan cuántos decimales o el símbolo de porcentaje se muestran.
formato_b = {
    'Corrección (0-2)': '{:.2f}',
    '% Fieles': '{:.1f}%',
    'Falsas Abstenciones': '{:d}',
    '% Formato e Idioma': '{:.1f}%'
}
display(df_b_view.style.format(formato_b).hide(axis='index'))

display(Markdown('### Tabla C. LLMs: Coste y Velocidad de Inferencia'))
# Hacemos lo mismo con las medidas de tiempo y memoria.
df_coste = pd.DataFrame(tabla_c_coste)
df_c_view = df_coste[[
    'modelo', 'stack', 'ttft_s', 'tokens_per_s', 'latencia_s', 'vram_gb'
]].copy()
df_c_view.columns = [
    'Modelo', 'Stack Inferencia', 'TTFT (s)', 'Tokens/s', 'Latencia Total (s)', 'VRAM Pico (GB)'
]
formato_c = {
    'TTFT (s)': '{:.3f} s',
    'Tokens/s': '{:.1f} tok/s',
    'Latencia Total (s)': '{:.2f} s',
    'VRAM Pico (GB)': '{:.2f} GB'
}
display(df_c_view.style.format(formato_c).hide(axis='index'))

### Tabla B. LLMs: Calidad de Respuestas RAG

Modelo,Tamaño,Cuantización,Corrección (0-2),% Fieles,Abstención Correcta,Falsas Abstenciones,% Formato e Idioma
Gemma-2-2B,2.6B,Q4_0,1.70,98.3%,11/12,6,98.3%
Granite-3-2B,2.6B,Q4_K_M,1.68,98.3%,11/12,4,75.0%
Mistral-7B,7.2B,Q4_K_M,1.75,100.0%,12/12,6,100.0%
Qwen-2.5-7B,7.6B,Q4_K_M,1.73,100.0%,12/12,8,100.0%


### Tabla C. LLMs: Coste y Velocidad de Inferencia

Modelo,Stack Inferencia,TTFT (s),Tokens/s,Latencia Total (s),VRAM Pico (GB)
Gemma-2-2B,Ollama,0.044 s,86.7 tok/s,0.44 s,1.52 GB
Granite-3-2B,Ollama,0.015 s,98.9 tok/s,0.41 s,1.49 GB
Mistral-7B,Ollama,0.016 s,72.9 tok/s,0.68 s,4.07 GB
Qwen-2.5-7B,Ollama,0.014 s,74.0 tok/s,0.43 s,4.36 GB


## 6. Desglose por Documento: Normativa (BOE) vs Técnica (Debian)

In [7]:
# Agrupamos las respuestas por documento y modelo para sacar medias y totales.
df_respuestas = pd.DataFrame(respuestas_detalladas)
# Sacamos cada métrica del diccionario de evaluación para poder agruparla con pandas.
df_respuestas['correccion'] = df_respuestas['evaluacion'].apply(lambda x: x['correccion'])
df_respuestas['fiel'] = df_respuestas['evaluacion'].apply(lambda x: 100.0 if x['fiel'] else 0.0)
df_respuestas['falsa_abs'] = df_respuestas['evaluacion'].apply(lambda x: 1 if x['falsa_abstencion'] else 0)
df_respuestas['formato'] = df_respuestas['evaluacion'].apply(lambda x: 100.0 if x['formato_idioma'] else 0.0)

# Calculamos el resumen de las métricas para cada grupo.
resumen_doc = df_respuestas.groupby(['documento', 'modelo']).agg({
    'correccion': 'mean',
    'fiel': 'mean',
    'falsa_abs': 'sum',
    'formato': 'mean'
}).reset_index()

# Sustituimos los identificadores cortos por nombres completos para mostrarlos.
nombres_doc = {
    'boe': '1. Normativa (BOE Estatuto de los Trabajadores)',
    'debian': '2. Técnica (Debian Handbook)'
}
resumen_doc['documento'] = resumen_doc['documento'].map(nombres_doc)
resumen_doc.columns = ['Documento', 'Modelo', 'Corrección (0-2)', '% Fieles', 'Falsas Abstenciones', '% Formato']

# Separamos el resumen por documento y mostramos una tabla para cada uno.
for doc_name, grupo in resumen_doc.groupby('Documento'):
    display(Markdown(f'### {doc_name}'))
    grupo_clean = grupo[['Modelo', 'Corrección (0-2)', '% Fieles', 'Falsas Abstenciones', '% Formato']]
    styler = grupo_clean.style.format({
        'Corrección (0-2)': '{:.2f}',
        '% Fieles': '{:.1f}%',
        'Falsas Abstenciones': '{:d}',
        '% Formato': '{:.1f}%'
    }).hide(axis='index')
    display(styler)

### 1. Normativa (BOE Estatuto de los Trabajadores)

Modelo,Corrección (0-2),% Fieles,Falsas Abstenciones,% Formato
Gemma-2-2B,1.83,96.7%,1,100.0%
Granite-3-2B,1.73,96.7%,2,80.0%
Mistral-7B,1.87,100.0%,1,100.0%
Qwen-2.5-7B,1.93,100.0%,1,100.0%


### 2. Técnica (Debian Handbook)

Modelo,Corrección (0-2),% Fieles,Falsas Abstenciones,% Formato
Gemma-2-2B,1.57,100.0%,5,96.7%
Granite-3-2B,1.63,100.0%,2,70.0%
Mistral-7B,1.63,100.0%,5,100.0%
Qwen-2.5-7B,1.53,100.0%,7,100.0%


## 7. Ejemplos Reales de Respuestas

Vemos tres ejemplos concretos (una pregunta del BOE, una de Debian y una que no está en el documento) para comprobar cómo contesta cada modelo en la práctica.

In [8]:
# Mostramos algunas preguntas concretas y las respuestas que dio cada modelo.
preguntas_ejemplo = ['boe_q01', 'deb_q01', 'boe_q13']

for pid in preguntas_ejemplo:
    # Filtramos las respuestas detalladas para quedarnos con una pregunta concreta.
    items = [x for x in respuestas_detalladas if x['pregunta_id'] == pid]
    if not items:
        continue
    it0 = items[0]
    display(Markdown(f"---\n**Pregunta [{it0['pregunta_id']} - Tipo: {it0['tipo']}]:** {it0['pregunta']}"))
    display(Markdown(f"**Respuesta esperada:** *{it0['respuesta_esperada']}*"))

    # Preparamos una fila por modelo para comparar sus respuestas a esa pregunta.
    filas_ej = []
    for it in items:
        filas_ej.append({
            'Modelo': it['modelo'],
            'Respuesta Generada': it['respuesta_modelo'].strip(),
            'Corrección (0-2)': it['evaluacion']['correccion']
        })
    df_ej = pd.DataFrame(filas_ej)
    display(df_ej.style.hide(axis='index'))

---
**Pregunta [boe_q01 - Tipo: directa]:** ¿Cuántos días de vacaciones anuales retribuidas corresponden por ley como mínimo a un trabajador?

**Respuesta esperada:** *Como mínimo corresponden treinta días naturales al año (artículo 38.1).*

Modelo,Respuesta Generada,Corrección (0-2)
Gemma-2-2B,"Fuente: [Artículo 38. Vacaciones anuales. (Parte 1)] Respuesta: El periodo de vacaciones anuales retribuidas, no sustituible por compensación económica, será el pactado en convenio colectivo o contrato individual. En ningún caso la duración será inferior a treinta días naturales.",2
Granite-3-2B,"Fuente: [Artículo 38. Vacaciones anuales. (Parte 1)] Respuesta: Según la ley, los trabajadores tienen derecho a un mínimo de treinta días naturales de vacaciones anuales retribuidas.",2
Mistral-7B,"Fuente: [Artículo 38. Vacaciones anuales. (Parte 1)] Respuesta: El periodo de vacaciones anuales retribuidas, no sustituible por compensación económica, será el pactado en convenio colectivo o contrato individual. En ningún caso la duración será inferior a treinta días naturales. (No se especifica un mínimo legal en el documento proporcionado.)",2
Qwen-2.5-7B,"Fuente: [Artículo 38. Vacaciones anuales. - Parte 1/3] Respuesta: Según la ley, el periodo de vacaciones anuales retribuidas mínimo es de treinta días naturales.",2


---
**Pregunta [deb_q01 - Tipo: directa]:** ¿Qué comando y parámetro de bajo nivel se utiliza para instalar un archivo local de paquete Debian (.deb) mediante la herramienta dpkg?

**Respuesta esperada:** *Se utiliza el comando «dpkg -i nombre_paquete.deb» (o «dpkg --install») (Sección 5.4).*

Modelo,Respuesta Generada,Corrección (0-2)
Gemma-2-2B,Fuente: [5.4.1. Instalación de paquetes] Respuesta: dpkg -i paquete.deb,2
Granite-3-2B,Fuente: [5.4. Manipulación de paquetes con dpkg (Parte 1)] Respuesta: dpkg -i paquete.deb,2
Mistral-7B,Fuente: [5.4.1. Instalación de paquetes] Respuesta: El comando y parámetro de bajo nivel que se utiliza para instalar un archivo local de paquete Debian (.deb) mediante la herramienta dpkg es -i o --install.,2
Qwen-2.5-7B,Fuente: [5.4.1. Instalación de paquetes] Respuesta: Se utiliza el comando `dpkg` con el parámetro `-i` o `--install` para instalar un archivo local de paquete Debian (.deb).,2


---
**Pregunta [boe_q13 - Tipo: sin_respuesta]:** ¿A cuántos euros asciende exactamente el importe mensual del Salario Mínimo Interprofesional fijado para este año?

**Respuesta esperada:** *No aparece en el documento. El artículo 27 dice que el Gobierno fija anualmente el salario mínimo previa consulta sindical, pero no incluye cifras concretas en euros.*

Modelo,Respuesta Generada,Corrección (0-2)
Gemma-2-2B,Fuente: [Artículo 27. Salario mínimo interprofesional. (Parte 1)] Respuesta: No aparece en el documento.,2
Granite-3-2B,No aparece en el documento.,2
Mistral-7B,Fuente: [Artículo 27. Salario mínimo interprofesional. (Parte 1)] Respuesta: No aparece en el documento que se proporcionó información específica sobre el importe mensual del Salario Mínimo Interprofesional fijado para este año.,2
Qwen-2.5-7B,No aparece en el documento.,2


## 8. Conclusiones, Elección del Grupo y Plan B (8 GB de VRAM)

### Respuestas rápidas a las preguntas del benchmark (60 preguntas totales)
- **Mejor relación calidad/velocidad:** **`Gemma-2-2B`** (1.70/2.0, 98.3% de formato, 86.7 tok/s, 0.44 s y solo 1.52 GB de VRAM) y **`Granite-3-2B`** (1.68/2.0, el más rápido con 98.9 tok/s, 0.41 s y 1.49 GB de VRAM).
- **¿Compensa pasar de ~2.6B a ~7B?** Sí cuando la prioridad es no alucinar nunca: solo los dos modelos de 7B alcanzan **100% de fidelidad**, **12/12 en abstención correcta** y **100% de formato**. Además, en el BOE (`Qwen-2.5-7B` con 1.93 y `Mistral-7B` con 1.87) superan con claridad a los pequeños.
- **¿Quién alucina menos cuando no está en el texto?** **`Mistral-7B`** y **`Qwen-2.5-7B`**: **12/12 en abstención correcta** y **100% de fidelidad** (los de 2.6B aciertan 11/12 y 98.3% de fidelidad).

### Selección de ganadores
- **Para el Documento 1 (BOE):** Elegimos **`Qwen-2.5-7B`** porque obtiene la puntuación más alta en normativa (**1.93/2.00**), 100% de fidelidad, 6/6 en abstención, 100% de formato y solo 1 falsa abstención.
- **Para el Documento 2 (Debian):** Elegimos **`Granite-3-2B`** porque empata en el primer puesto de documentación técnica (**1.63/2.00**) cometiendo muy pocas falsas abstenciones (solo 2 frente a 5–7 de los de 7B), a 98.9 tok/s y con solo 1.49 GB de VRAM.
- **Decisión global del grupo:** Como único modelo para todo el sistema nos quedamos con **`Mistral-7B`**. Al trabajar con leyes y administración de sistemas, **priorizamos la fiabilidad total y no inventar nunca un dato antes que ahorrar algo de velocidad o memoria**. Con 60 preguntas es el modelo con **mayor corrección media global (1.75/2.00)**, **100% de fidelidad**, **12/12 en abstención**, **100% de formato**, menos falsas abstenciones que Qwen (6 frente a 8), menos VRAM (4.07 GB) y rinde muy bien en ambos documentos (1.87 en el BOE y 1.63 empatado en cabeza en Debian).

### Plan B: Servidor con 8 GB de VRAM
En un servidor limitado a 8 GB de VRAM, `Multilingual-E5-Small` (1.80 GB) + `Mistral-7B` (4.07 GB) suma **5.87 GB**, dejando poco margen si entran varias consultas a la vez. Como **Plan B** bajaríamos el LLM a **`Gemma-2-2B`** (1.52 GB): junto al embedding consume solo **3.32 GB de VRAM**, mantiene una calidad muy cercana (**1.70/2.00** y **98.3% de formato**) y deja casi 5 GB libres.